# Initialization

In [1]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# Загрузка данных

In [3]:
items = pd.read_parquet("items.par")
events = pd.read_parquet("events.par")

# Разбиение с учётом хронологии

Рекомендательные системы на практике работают с учётом хронологии. Поэтому поток событий для тренировки и валидации полезно делить на то, что уже случилось, и что ещё случится. Это позволяет проводить валидацию на тех же пользователях, на которых тренировались, но на их событиях в будущем.

# === Знакомство: "холодный" старт

In [4]:
# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01").date()

train_test_global_time_split_idx = events["started_at"] < train_test_global_time_split_date
events_train = events[train_test_global_time_split_idx] # ваш код здесь #
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates() # ваш код здесь #

# количество пользователей, которые есть и в train, и в test
common_users = np.intersect1d(users_train, users_test)# ваш код здесь #

print(len(users_train), len(users_test), len(common_users))

428220 123223 120858


«Холодные» пользователи — те, которые есть в test, но отсутствуют в train. Это соответствует хронологическому порядку, в котором и работает рекомендательная система.

In [5]:
cold_users = users_test[~users_test.isin(users_train)] # ваш код здесь #

print(len(cold_users)) 

2365


# === Знакомство: первые персональные рекомендации

In [6]:
from sklearn.preprocessing import MinMaxScaler

top_pop_start_date = pd.to_datetime("2015-01-01").date()

item_popularity = events_train \
    .query("started_at >= @top_pop_start_date") \
    .groupby(["item_id"]).agg(users=("user_id", "nunique"), avg_rating=("rating", "mean")).reset_index()

# нормализация пользователей и среднего рейтинга, требуется для их приведения к одному масштабу
scaler = MinMaxScaler()
item_popularity[["users_norm", "avg_rating_norm"]] = scaler.fit_transform(
    item_popularity[["users", "avg_rating"]]
)

# вычисляем popularity_score, как скор популярности со штрафом за низкий рейтинг
item_popularity["popularity_score"] = (
    item_popularity["users_norm"] * item_popularity["avg_rating_norm"]
)

# сортируем по убыванию popularity_score
item_popularity = item_popularity.sort_values("popularity_score", ascending=False) # ваш код здесь #

# выбираем первые 100 айтемов со средней оценкой avg_rating не меньше 4
top_k_pop_items = item_popularity[item_popularity["avg_rating"] >= 4].head(100) # ваш код здесь # 

# Сколько пользователей оценило книгу, попавшую на первое место в top_k_pop_items?
first_item_users = top_k_pop_items.iloc[0]['users']
print(f'total users {first_item_users}')

first_item = top_k_pop_items.iloc[0]
print(f"Самая популярная книга (item_id={first_item['item_id']})")
print(f"Средний рейтинг: {first_item['avg_rating']:.2f}")
print(f"Количество оценивших пользователей: {first_item['users']}")

# добавляем информацию о книгах
top_k_pop_items = top_k_pop_items.merge(
    items.set_index("item_id")[["author", "title", "genre_and_votes", "publication_year"]], on="item_id")

with pd.option_context('display.max_rows', 10):
    display(top_k_pop_items[["item_id", "author", "title", "publication_year", "users", "avg_rating", "popularity_score", "genre_and_votes"]]) 
    

total users 20207.0
Самая популярная книга (item_id=18007564.0)
Средний рейтинг: 4.32
Количество оценивших пользователей: 20207.0


,item_id,author,title,publication_year,users,avg_rating,popularity_score,genre_and_votes
0,18007564,Andy Weir,The Martian,2014,20207,4.321275,0.412333,"{'Science Fiction': 11966, 'Fiction': 8430}"
1,18143977,Anthony Doerr,All the Light We Cannot See,2014,19462,4.290669,0.393471,"{'Historical-Historical Fiction': 13679, 'Fict..."
2,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,1997,15139,4.706057,0.344702,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad..."
3,16096824,Sarah J. Maas,A Court of Thorns and Roses (A Court of Thorns...,2015,16770,4.301014,0.340108,"{'Fantasy': 14326, 'Young Adult': 4662, 'Roman..."
4,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,1999,13043,4.632447,0.291076,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict..."
...,...,...,...,...,...,...,...,...
95,8490112,Laini Taylor,Daughter of Smoke & Bone (Daughter of Smoke & ...,2011,4792,4.080968,0.090694,"{'Fantasy': 11681, 'Young Adult': 7110, 'Roman..."
96,18966819,Pierce Brown,"Golden Son (Red Rising, #2)",2015,4361,4.374914,0.090409,"{'Science Fiction': 2613, 'Fantasy': 1372, 'Sc..."
97,3636,Lois Lowry,"The Giver (The Giver, #1)",2006,4667,4.098564,0.088832,"{'Young Adult': 10993, 'Fiction': 9045, 'Class..."
98,18293427,Gabrielle Zevin,The Storied Life of A.J. Fikry,2014,4674,4.092640,0.088795,"{'Fiction': 3795, 'Contemporary': 1100, 'Writi..."


Завершите предлагаемый код, чтобы в cold_users_events_with_recs для каждого события получить столбец avg_rating. В нём при совпадении по item_id будет значение из одноимённого столбца из top_k_pop_items, иначе — пропуск.
В cold_users_events_with_recs после выполнения завершённого кода должно быть столько же строк, сколько было до его выполнения.

In [7]:

cold_users_events_with_recs = \
    events_test[events_test["user_id"].isin(cold_users)] \
    .merge(
        top_k_pop_items[["item_id", "avg_rating"]], # ваш код здесь # 
        on="item_id", 
        how="left")

cold_user_items_no_avg_rating_idx = cold_users_events_with_recs["avg_rating"].isnull()
cold_user_recs = cold_users_events_with_recs[~cold_user_items_no_avg_rating_idx] \
    [["user_id", "item_id", "rating", "avg_rating"]]

# Вычисляем долю событий холодных пользователей, для которых книга есть в топ-100
total_cold_events = len(cold_users_events_with_recs)
matched_events = (~cold_user_items_no_avg_rating_idx).sum()
ratio = matched_events / total_cold_events
print(round(ratio, 2))
print('cold_users_events_with_recs 2: ', len(cold_users_events_with_recs))

# Посчитайте метрики rmse и mae для полученных рекомендаций.
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = mean_squared_error(cold_user_recs["rating"], cold_user_recs["avg_rating"]) ** 0.5
mae = mean_absolute_error(cold_user_recs["rating"], cold_user_recs["avg_rating"])
print(round(rmse, 2), round(mae, 2))

# посчитаем покрытие холодных пользователей рекомендациями

cold_users_hit_ratio = cold_users_events_with_recs.groupby("user_id").agg(hits=("avg_rating", lambda x: (~x.isnull()).mean()))

print(f"Доля пользователей без релевантных рекомендаций: {(cold_users_hit_ratio == 0).mean().iat[0]:.2f}")
print(f"Среднее покрытие пользователей: {cold_users_hit_ratio[cold_users_hit_ratio != 0].mean().iat[0]:.2f}") 


0.2
cold_users_events_with_recs 2:  9672
0.78 0.62
Доля пользователей без релевантных рекомендаций: 0.59
Среднее покрытие пользователей: 0.44


# === Базовые подходы: коллаборативная фильтрация

In [8]:
import scipy
import sklearn.preprocessing

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["item_id"])
items["item_id_enc"] = item_encoder.transform(items["item_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"])# ваш код здесь #
events_test["item_id_enc"] = item_encoder.transform(events_test["item_id"])# ваш код здесь # 

# Вывод максимального значения закодированного идентификатора объекта в обучающей выборке
max_encoded_item = events_train["item_id_enc"].max()
print(f"Максимальное значение item_id_enc в events_train: {max_encoded_item}")

Максимальное значение item_id_enc в events_train: 43304


In [9]:
# Количество уникальных пользователей в обучающей выборке
num_users = len(users_train)

# Количество уникальных книг (всех предметов)
num_items = len(items)

# Общее количество элементов матрицы (включая пропуски)
total_elements = num_users * num_items

# Размер в байтах (1 байт на элемент), переводим в гигабайты (целая часть)
size_gb = total_elements // (1024 ** 3)

print(size_gb)

17


Матрица взаимодействий — сильно разреженная, и можно хранить её эффективнее. Рассматриваемая реализация ALS работает с numpy-матрицами, поэтому её можно создать с помощью методов numpy. Мы создадим sparse-матрицу в формате CSR. Такая матрица хранит только имеющиеся значения и не резервирует память под пропуски. Это помогает значительно сэкономить оперативную память при работе с разреженными матрицами. Подробнее разобраться со sparse-матрицами в numpy можно на странице документации.

In [10]:
# создаём sparse-матрицу формата CSR 
user_item_matrix_train = scipy.sparse.csr_matrix((
    events_train["rating"],
    (events_train['user_id_enc'], events_train['item_id_enc'])),
    dtype=np.int8) 

sparse-формат numpy-матриц позволяет сильно уменьшить требование к размеру памяти. Можно самостоятельно посчитать и сравнить, используя код ниже:

In [11]:
import sys

sum([sys.getsizeof(i) for i in user_item_matrix_train.data])/1024**3 

# Размер плотной матрицы (в ГБ, целая часть)
dense_size_gb = (len(users_train) * len(items)) // (1024 ** 3)

# Разреженная матрица хранит:
# - data: ненулевые значения (тип int8, 1 байт)
# - indices: индексы столбцов (тип int32, 4 байта)
# - indptr: указатели на строки (тип int32, 4 байта)
sparse_data_bytes = user_item_matrix_train.data.nbytes
sparse_indices_bytes = user_item_matrix_train.indices.nbytes
sparse_indptr_bytes = user_item_matrix_train.indptr.nbytes
sparse_total_bytes = sparse_data_bytes + sparse_indices_bytes + sparse_indptr_bytes
sparse_size_gb = sparse_total_bytes / (1024 ** 3)

print(f"Размер плотной матрицы: {dense_size_gb} ГБ")
print(f"Размер разреженной матрицы: {sparse_size_gb:.6f} ГБ")
print(f"Экономия памяти: {dense_size_gb / sparse_size_gb:.1f}x")


Размер плотной матрицы: 17 ГБ
Размер разреженной матрицы: 0.054345 ГБ
Экономия памяти: 312.8x


Имея подготовленную матрицу взаимодействий, перейдём к третьему шагу — создадим ALS-модель. Для примера возьмём количество латентных факторов для матриц $P, Q$, равным 50. 
Выполните код для создания и тренировки модели.

In [12]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05, random_state=0)
als_model.fit(user_item_matrix_train) 

/home/ausachev/work/yandex.practicum/sprint4/mle-recsys-start/env_recsys_start/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/50 [00:00<?, ?it/s]

Чтобы получить рекомендации для пользователя с помощью модели ALS, используем такую функцию:

In [13]:
def get_recommendations_als(user_item_matrix, model, user_id, user_encoder, item_encoder, include_seen=True, n=5):
    """
    Возвращает отранжированные рекомендации для заданного пользователя
    """
    user_id_enc = user_encoder.transform([user_id])[0]
    recommendations = model.recommend(
         user_id_enc, 
         user_item_matrix[user_id_enc], 
         filter_already_liked_items=not include_seen,
         N=n)
    recommendations = pd.DataFrame({"item_id_enc": recommendations[0], "score": recommendations[1]})
    recommendations["item_id"] = item_encoder.inverse_transform(recommendations["item_id_enc"])
    
    return recommendations 

In [ ]:
# получаем список всех возможных user_id (перекодированных)
user_ids_encoded = range(len(user_encoder.classes_))

# получаем рекомендации для всех пользователей
als_recommendations = als_model.recommend(
    user_ids_encoded, 
    user_item_matrix_train[user_ids_encoded], 
    filter_already_liked_items=False, N=100) 

# преобразуем полученные рекомендации в табличный формат
item_ids_enc = als_recommendations[0]
als_scores = als_recommendations[1]

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(), 
    "score": als_scores.tolist()})
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")

# получаем изначальные идентификаторы
als_recommendations["user_id"] = user_encoder.inverse_transform(als_recommendations["user_id_enc"])
als_recommendations["item_id"] = item_encoder.inverse_transform(als_recommendations["item_id_enc"])
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"])

als_recommendations = als_recommendations[["user_id", "item_id", "score"]]
als_recommendations.to_parquet("als_recommendations.parquet") 

### Метрики
Score от ALS не лежат на той же шкале, что и пользовательские оценки. Сравнивать исходные и новые оценки напрямую — некорректно. Поэтому посчитать метрики MAE, RMSE проблематично. Вместо них можно использовать метрики ранжирования. Они сравнивают не абсолютные значения рейтингов и их оценок, а соответствие порядков. Метрики ранжирования покажут, насколько порядок рекомендаций по убыванию score соответствует порядку объектов по убыванию пользовательских оценок. 
На практике часто используют метрику NDCG, она принимает значение от 0 (предлагаемый порядок никак не соответствует истинному) до 1 (предлагаемый порядок в точности соответствует истинному). 


In [ ]:
als_recommendations = (
    als_recommendations
    .merge(events_test[["user_id", "item_id", "rating"]]
               .rename(columns={"rating": "rating_test"}), 
           on=["user_id", "item_id"], how="left")
)

import sklearn.metrics

def compute_ndcg(rating: pd.Series, score: pd.Series, k):

    """ подсчёт ndcg
    rating: истинные оценки
    score: оценки модели
    k: количество айтемов (по убыванию score) для оценки, остальные - отбрасываются
    """
    
    # если кол-во объектов меньше 2, то NDCG - не определена
    if len(rating) < 2:
        return np.nan

    ndcg = sklearn.metrics.ndcg_score(np.asarray([rating.to_numpy()]), np.asarray([score.to_numpy()]), k=k)

    return ndcg 

# Умея считать NDCG для одного пользователя, посчитаем данную метрику, например, для k=5 для всех пользователей из тестовой выборки. 
# В результате каждому пользователю будет соответствовать одно значение NDCG@5. 
# Запись “NDCG@5” означает, что метрика NDCG считается для пяти объектов. 

rating_test_idx = ~als_recommendations["rating_test"].isnull()
ndcg_at_5_scores = als_recommendations[rating_test_idx].groupby("user_id").apply(lambda x: compute_ndcg(x["rating_test"], x["score"], k=5))

# Имея ряд значений NDCG@5 по пользователям, можно посчитать среднее её значение (по всем пользователям):
print(ndcg_at_5_scores.mean()) 

# === Базовые подходы: контентные рекомендации

Составим список всех жанров
Составим список всех возможных жанров по всем айтемам. Потом по нему будем составлять вектор с весами по жанрам для каждой книги. Из любопытства составим список жанров с подсчётом количества голосов по каждому из них.
Сначала преобразуем значения в genre_and_votes из текстового представления в тип в Python:

In [ ]:
# преобразование в dict тип
items["genre_and_votes"] = items["genre_and_votes"].apply(eval)

# проверка
print(type(items["genre_and_votes"].iloc[0]))


In [ ]:
def get_genres(items):

    """ 
    извлекает список жанров по всем книгам, 
    подсчитывает долю голосов по каждому их них
    """
    
    genres_counter = {}
    
    for k, v, in items.iterrows():
        genre_and_votes = v["genre_and_votes"] # ваш код здесь #
        if genre_and_votes is None or not isinstance(genre_and_votes, dict):
            continue
        for genre, votes in genre_and_votes.items():
            # увеличиваем счётчик жанров
            try:
                genres_counter[genre] += votes # ваш код здесь #
            except KeyError:
                genres_counter[genre] = 0

    genres = pd.Series(genres_counter, name="votes")
    genres = genres.to_frame()
    genres = genres.reset_index().rename(columns={"index": "name"})
    genres.index.name = "genre_id"
    
    return genres
   
genres = get_genres(items)

genres["score"] = genres["votes"] / genres["votes"].sum()
genres_sorted = genres.sort_values(by="score", ascending=False).head(10) 

# Пятый по популярности жанр (индекс 4)
if len(genres_sorted) >= 5:
    fifth = genres_sorted.iloc[4]["name"]
    print(f"Пятый по популярности жанр: {fifth}")
else:
    print("Жанров меньше пяти")

In [ ]:
def get_item2genre_matrix(genres, items):
    '''
    Функция в коде ниже строит матрицу вида «книга-жанр». 
    '''
    
    genre_names_to_id = genres.reset_index().set_index("name")["genre_id"].to_dict()
    
    # list to build CSR matrix
    genres_csr_data = []
    genres_csr_row_idx = []
    genres_csr_col_idx = []
    
    for item_idx, (k, v) in enumerate(items.iterrows()):
        if v["genre_and_votes"] is None:
            continue
        for genre_name, votes in v["genre_and_votes"].items():
            genre_idx = genre_names_to_id[genre_name]
            genres_csr_data.append(int(votes))
            genres_csr_row_idx.append(item_idx)
            genres_csr_col_idx.append(genre_idx)

    genres_csr = scipy.sparse.csr_matrix((genres_csr_data, (genres_csr_row_idx, genres_csr_col_idx)), shape=(len(items), len(genres)))
    # нормализуем, чтобы сумма оценок принадлежности к жанру была равна 1
    genres_csr = sklearn.preprocessing.normalize(genres_csr, norm='l1', axis=1)
    
    return genres_csr 

Получим матрицу с весами по жанрам для каждой книги:


In [ ]:
items = items.sort_values(by="item_id_enc")
all_items_genres_csr = get_item2genre_matrix(genres, items) 

Это важный результат. Он поможет вычислить, какие книги «ближе» к пользовательскому вектору.
Аналогичным образом получим матрицу с весами по жанрам для какого-нибудь пользователя, например, для пользователя с идентификатором 1000010.

In [ ]:
user_id = 1000010
user_events = events_train.query("user_id == @user_id")[["item_id", "rating"]]
user_items = items[items["item_id"].isin(user_events["item_id"])]

user_items_genres_csr = get_item2genre_matrix(genres, user_items)
# Количество существующих элементов (ненулевых)
print(user_items_genres_csr.nnz)

Если посчитать средние, то фактически получим предпочтения пользователя по жанрам.
На практике часто пользователь явно указывает предпочтения в своём профиле. У нас таких данных нет, поэтому предпочтения пользователя по жанрам вычислим автоматически на основе его истории поведения. 

In [ ]:
# вычислим склонность пользователя к жанрам как среднее взвешенное значение популяции на его оценки книг.

# преобразуем пользовательские оценки из списка в вектор-столбец
user_ratings = user_events["rating"].to_numpy() / 5
user_ratings = np.expand_dims(user_ratings, axis=1)

user_items_genres_weighted = user_items_genres_csr.multiply(user_ratings)

user_genres_scores = np.asarray(user_items_genres_weighted.mean(axis=0))

# выведем список жанров, которые предпочитает пользователь

user_genres = genres.copy()
user_genres["score"] = np.ravel(user_genres_scores)
user_genres = user_genres[user_genres["score"] > 0].sort_values(by=["score"], ascending=False)

user_genres.head(5) 

Построим рекомендации
Теперь рассчитаем рекомендации на основе двух объектов:
all_items_genres_csr — матрица распределения интересов всех пользователей по жанрам. Строка — вектор распределения интересов всех пользователей по жанрам для одного объекта.
user_genres_scores — вектор интересов пользователя по жанрам.
Рекомендации будем рассчитывать с помощью косинусного сходства между двумя векторами

где 
v — вектор-строка из all_items_genres_csr, 
w — вектор-столбец из user_genres_scores. 
Чтобы посчитать косинусное сходство user_genres_scores к вектор-строкам из all_items_genres_csr, необходимо выполнить вычисления по формуле столько раз, сколько есть строк в all_items_genres_csr.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# вычисляем сходство между вектором пользователя и векторами книг
similarity_scores = cosine_similarity(all_items_genres_csr, user_genres_scores)

# преобразуем в одномерный массив
similarity_scores = similarity_scores.flatten()

# получаем индексы top-5 книг (по убыванию сходства)
k = 5
top_k_indices = np.argsort(similarity_scores)[::-1][:k]

# отбираем книги с этими индексами
selected_items = items[items["item_id_enc"].isin(top_k_indices)]

# выводим информацию о рекомендованных книгах
with pd.option_context("max_colwidth", 100):
    display(selected_items[["author", "title", "genre_and_votes"]])

# вычисляем сходство между вектором пользователя и векторами по книгам
similarity_scores = cosine_similarity(all_items_genres_csr, user_genres_scores)

# преобразуем в одномерный массив
similarity_scores = similarity_scores.flatten()

# получаем индексы top-k (по убыванию значений), по сути, индексы книг (encoded)
k = 5
top_k_indices = np.argsort(similarity_scores)[::-1][:k] # ваш код здесь #

selected_items = items[items["item_id_enc"].isin(top_k_indices)]
with pd.option_context("max_colwidth", 100):
    display(selected_items[["author", "title", "genre_and_votes"]])

In [ ]:
from collections import Counter

# Собираем все жанры из рекомендованных книг (учитываем только название, без весов)
genres_list = []
for _, row in selected_items.iterrows():
    if isinstance(row["genre_and_votes"], dict):
        genres_list.extend(row["genre_and_votes"].keys())

# Подсчитываем частоту каждого жанра
genre_counter = Counter(genres_list)
print("Жанры в рекомендациях с частотами:")
print(genre_counter.most_common())

# Самый частый жанр
if genre_counter:
    most_common_genre = genre_counter.most_common(1)[0][0]
    print(f"Преобладающий жанр: {most_common_genre}")
else:
    print("Нет данных о жанрах в рекомендациях.")

# === Базовые подходы: валидация

### Метрики классификации («попадания»)
Используются метрики, традиционные для задач бинарной классификации: precision, recall, и их производные, например, F1.

#### Precision
Метрика показывает, какая доля объектов, отнесённых моделью к положительному классу, действительно относится к этому классу. В контексте задачи рекомендаций это означает, что рекомендованный айтем действительно нравится пользователю (есть в тестовой выборке).

#### Recall
Recall измеряет способность модели обнаруживать все истинные случаи в данных. В контексте задачи рекомендаций это означает, что модель смогла «найти» те айтемы, которые представлены в тестовой выборке.

Посчитаем recall и precision для ALS-рекомендаций (als_recommendations). Для этого события в тестовой выборке и рекомендации для одних и тех же пользователей разметим признаками:
- gt (ground truth): объект есть в тестовой выборке;
- pr (predicted): объект есть в рекомендациях.

Теперь разметим признаки бинарной классификации:
- TP: объект есть и в тестовой выборке, и в рекомендациях (истинная рекомендация),
- FP: объекта нет в тестовой выборке, но он есть в рекомендациях (ложноположительная рекомендация),
- FN: объект есть в тестовой выборке, но его нет в рекомендациях (ложноотрицательная рекомендация)
    

In [ ]:
def process_events_recs_for_binary_metrics(events_train, events_test, recs, top_k=None):

    """
    размечает пары <user_id, item_id> для общего множества пользователей признаками
    - gt (ground truth)
    - pr (prediction)
    top_k: расчёт ведётся только для top k-рекомендаций
    """

    events_test["gt"] = True
    common_users = set(events_test["user_id"]) & set(recs["user_id"])

    print(f"Common users: {len(common_users)}")
    
    events_for_common_users = events_test[events_test["user_id"].isin(common_users)].copy()
    recs_for_common_users = recs[recs["user_id"].isin(common_users)].copy()

    recs_for_common_users = recs_for_common_users.sort_values(["user_id", "score"], ascending=[True, False])

    # оставляет только те item_id, которые были в events_train, 
    # т. к. модель не имела никакой возможности давать рекомендации для новых айтемов
    events_for_common_users = events_for_common_users[events_for_common_users["item_id"].isin(events_train["item_id"].unique())]

    if top_k is not None:
        recs_for_common_users = recs_for_common_users.groupby("user_id").head(top_k)
    
    events_recs_common = events_for_common_users[["user_id", "item_id", "gt"]].merge(
        recs_for_common_users[["user_id", "item_id", "score"]], 
        on=["user_id", "item_id"], how="outer")    

    events_recs_common["gt"] = events_recs_common["gt"].fillna(False)
    events_recs_common["pr"] = ~events_recs_common["score"].isnull()
    
    events_recs_common["tp"] = events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fp"] = ~events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fn"] = events_recs_common["gt"] & ~events_recs_common["pr"]

    return events_recs_common

In [ ]:
events_recs_for_binary_metrics = process_events_recs_for_binary_metrics(
    events_train,
    events_test, 
    als_recommendations, 
    top_k=5) 

Мы получили разметку случившихся в будущем событий (events_test) и рекомендаций (als_recommendations: то есть событий, спрогнозированных как случившиеся в будущем). На их основе можно посчитать precision и recall по вышеприведённым формулам. Ниже дана заготовка функции, которая считает эти метрики. На вход ей подаётся результат выполнения функции process_events_recs_for_binary_metrics

In [ ]:
def compute_cls_metrics(events_recs_for_binary_metric):
    
    groupper = events_recs_for_binary_metric.groupby("user_id")

    # precision = tp / (tp + fp)
    precision = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fp"].sum())
    precision = precision.fillna(0).mean()
    
    # recall = tp / (tp + fn)
    recall = groupper["tp"].sum() / (groupper["tp"].sum() + groupper["fn"].sum())
    recall = recall.fillna(0).mean() # ваш код здесь #

    return precision, recall

In [ ]:
precision, recall = compute_cls_metrics(events_recs_for_binary_metrics)
print(f"precision@5 = {round(precision, 3)}, recall@5 = {round(recall, 3)}")

# === Двухстадийный подход: метрики

Для рекомендаций, сохранённых в переменной als_recommendations, посчитайте покрытие по объектам согласно формуле выше. При этом используйте весь топ-100 рекомендаций.

In [ ]:
cov_items = als_recommendations['item_id'].nunique() / len(items)
print(f"{cov_items:.2f}")

In [ ]:
# разметим каждую рекомендацию признаком read
events_train["read"] = True
als_recommendations = als_recommendations.merge(
    events_train[["user_id", "item_id", "read"]].drop_duplicates(), # ваш код здесь #, 
    on=["user_id", "item_id"], 
    how="left")
als_recommendations["read"] = als_recommendations["read"].fillna(False).astype("bool")

# проставим ранги
als_recommendations = als_recommendations.sort_values(["user_id", "score"], ascending=[True, False])
als_recommendations["rank"] = als_recommendations.groupby("user_id").cumcount() + 1

# посчитаем новизну novelty по пользователям
novelty_5 = (1-als_recommendations.query("rank <= 5").groupby("user_id")["read"].mean())

# посчитаем средний novelty
# ваш код здесь #
novelty_mean = novelty_5.mean()
print(f"Novelty@5 = {novelty_mean:.2f}")

# === Двухстадийный подход: модель

# === Двухстадийный подход: построение признаков